In [209]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from skimage.feature import hog
import tqdm
path1 = 'train_data (4)/'
path2 = 'test_data (2)/'
labels = np.zeros(2000, dtype=np.int64)
B = 64
X1 = np.zeros((2000, B, B, 3))
X2 = np.zeros((201, B, B, 3))
XX1 = np.zeros(2000)
XX2 = np.zeros(2000)
feats1 = []
feats2 = []
def extractfeatures(X:np.array):
    return X.std(), X[:,:,0].std()
for j in range(201):
    X2[j] = np.array(Image.open(path1 + 'img_'+str(j)+'.png').resize((B, B))) / 256
    XX2[j] = X2[j].sum()
    plt.imread(path1 + 'img_'+str(j)+'.png')
    feats2.append(extractfeatures(X2[j]))
    # print(X2[j])
for i in range(2000):
    X1[i] = np.array(Image.open(path2 + 'img_'+str(i)+'.png').resize((B, B))) / 256
    XX1[i] = X1[i].sum()
    plt.imread(path2 + 'img_'+str(i)+'.png')
    feats1.append(extractfeatures(X1[i]))
feats1 = np.array(feats1)
feats2 = np.array(feats2)



In [210]:
ids = []
for i in tqdm.tqdm(range(2000)):
    for j in range(201):
        if XX1[i] == XX2[j] and (X1[i] != X2[j]).sum() == 0:
            # print(j, i)
            labels[i] = 1
            ids.append(i)
            break
evts1 = []
for i in tqdm.tqdm(range(2000)):
    if i not in ids:
        sume = 0
        for j in ids:
            sume += np.abs(feats1[i] - feats1[j]).sum()
        evts1.append([-sume, i])
evts1.sort()
ids0 = []
for id in range(201):
    ids0.append(evts1[id][1])
ids0[:5]

100%|██████████| 2000/2000 [00:01<00:00, 1747.80it/s]


[964, 584, 381, 724, 68]

In [211]:
lX1 = []
lX2 = []
for t1 in ids:
    lX1.append(X1[t1])
for t2 in ids0:
    lX2.append(X1[t2])
lX1 = np.array(lX1)
lX2 = np.array(lX2)


In [212]:
model = LogisticRegression(max_iter=10000)
model.fit(np.concatenate([lX1, lX2]).reshape((402, -1)), np.concatenate([np.ones(201), np.zeros(201)]))
# model.predict()
final_labels = model.predict(X1.reshape((2000, -1)))
final_labels

array([1., 0., 0., ..., 1., 0., 0.], shape=(2000,))

In [213]:

import pandas as pd
res = pd.DataFrame({"subtaskID" : [1] * (2000), "datapointID" : range(2000), "answer" : np.int8(final_labels)})
res.to_csv('submission.csv', index=False)
res

,subtaskID,datapointID,answer
0,1,0,1
1,1,1,0
2,1,2,0
3,1,3,0
4,1,4,0
...,...,...,...
1995,1,1995,1
1996,1,1996,1
1997,1,1997,1
1998,1,1998,0
